# U2-8 · PubMed RCT BERT Embedding Comparison

**MSDS 565 · Unit 2 (Text Data) · Demo**

Clinical research papers follow a predictable rhetorical structure: *Background → Objective →
Methods → Results → Conclusions*. The **PubMed 20k RCT** dataset (Dernoncourt & Lee, 2017)
provides ~200,000 sentences from randomized controlled trials, each labeled with its section role.

> *Does pretraining on biomedical text actually help when classifying biomedical sentence roles?*

We answer by comparing **six frozen BERT models** — from general-purpose to PubMed-specific —
as feature extractors, training the same two classifiers (Logistic Regression and Random Forest)
on each embedding, and visualizing structure with **t-SNE per model**.

**Workflow**

| Step | Task |
|------|------|
| **Load** | Read the PubMed 20k RCT CSV files from disk; sample a balanced subset |
| **Explore** | Class balance, sentence-length distribution, example sentences per role |
| **Embed** | Mean-pool six frozen BERT models (general → bio → clinical → science) |
| **Classify** | Train LR + RF on each embedding; accuracy + macro-F1 leaderboard |
| **Visualize** | t-SNE per embedding colored by sentence role — does cluster separation track accuracy? |
| **Summarize** | Which pretraining corpus wins, and why? |

**Prediction target — five RCT sentence roles:**

| Label | Meaning | Example |
|-------|---------|---------|
| `BACKGROUND` | Context / motivation | *"Emotional eating is associated with overeating…"* |
| `OBJECTIVE` | Study aim | *"The aim of this study was to test…"* |
| `METHODS` | Study design / procedures | *"Participants were randomized 1:1…"* |
| `RESULTS` | Findings | *"There was a clinically relevant reduction…"* |
| `CONCLUSIONS` | Interpretation | *"Low-dose prednisolone had a sustained effect…"* |

---

**Dataset:** `PubMed_20k_RCT/train.csv`  (~180 k sentences, 5 balanced classes)
— Dernoncourt F & Lee JY. *PubMed 200k RCT.* arXiv:1710.06071 (2017)

**Prerequisites** (install once if not already present):
```
pip install transformers torch scikit-learn pandas numpy matplotlib
```
BERT model weights download on first run (~400 MB per model, then cached at
`~/.cache/huggingface/hub`).

## 1 · Imports

Standard libraries plus:
- **scikit-learn** — classifiers, train/test split, StandardScaler, evaluation metrics, t-SNE
- **transformers** — `pipeline("feature-extraction")` for frozen BERT embedding extraction

No `torch` imports needed directly — transformers handles the forward pass internally.
No web-scraping libraries needed — the dataset is already on disk.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
)
from sklearn.base import clone

from transformers import pipeline as hf_pipeline

pd.set_option("display.max_columns", 100)

## 2 · Load the PubMed 20k RCT Dataset

The dataset ships as three CSV files (`train.csv`, `dev.csv`, `test.csv`). Each row is a
**single sentence** from an RCT abstract with six columns:

| Column | Description |
|--------|-------------|
| `abstract_id` | PubMed ID of the source article |
| `line_id` | `{pmid}_{line_num}_{total_lines}` |
| `abstract_text` | The sentence text |
| `line_number` | Position of this sentence within the abstract (0-indexed) |
| `total_lines` | Total sentences in this abstract |
| `target` | Section label: `BACKGROUND`, `OBJECTIVE`, `METHODS`, `RESULTS`, `CONCLUSIONS` |

We load `train.csv` (the largest split) to maximize class balance. `dev.csv` and `test.csv`
are available if you want a held-out evaluation set that was never seen during embedding.

In [ ]:
DATA_DIR = r"C:\Users\Graham West\Python Notebooks\Meharry Teaching\Datasets\PubMed\PubMed_20k_RCT"

df = pd.read_csv(f"{DATA_DIR}/train.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution (target):")
print(df["target"].value_counts().to_string())
df.head(4)

## 3 · Configuration

`MAX_PER_CLS` controls how many sentences are drawn per class for the embedding experiment.
The full training set has ~180 k sentences, but embedding on CPU is slow (~2–3 s/sentence),
so we cap the per-class sample. Default `60` (300 total) takes ~10–20 min per BERT model
on CPU. Reduce to `30` for a quicker demo; increase to `150+` on GPU.

> The sample is drawn once from the training CSV and the same rows are reused for all six
> BERT models — only the embeddings change, not the underlying sentences.

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
MAX_PER_CLS  = 60    # sentences per class for embedding (60 × 5 = 300 total); raise on GPU
RANDOM_STATE = 42

CLASSES = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]

# ── Drop any rows with missing text or unknown label ─────────────────────────
df = df.dropna(subset=["abstract_text", "target"])
df = df[df["target"].isin(CLASSES)].reset_index(drop=True)

print(f"Usable rows: {len(df):,}")
print(df["target"].value_counts().to_string())

## 4 · Explore the Dataset

Before touching a model, we profile the sentences. Three questions:

1. **Balance** — are the five section roles roughly equally represented?
   (METHODS and RESULTS tend to dominate in RCT abstracts.)
2. **Length** — how many words per sentence? BERT handles up to 512 tokens; sentences
   are short, so truncation at `max_length=128` tokens should be safe.
3. **Examples** — what do representative sentences for each role look like?
   This confirms the MeSH queries returned topically coherent articles and that the
   labels are clean enough for classification.

In [ ]:
df["word_count"] = df["abstract_text"].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# -- 1. Sentences per class ---------------------------------------------------
counts = df["target"].value_counts().reindex(CLASSES)
counts.plot(kind="barh", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Sentences per Section Role")
axes[0].set_xlabel("Count")

# -- 2. Sentence word-count distribution --------------------------------------
axes[1].hist(df["word_count"], bins=40, color="darkorange", edgecolor="white")
axes[1].axvline(40, color="red", linestyle="--", linewidth=1,
                label="typical BERT 128-token limit ≈ 40 words")
axes[1].set_title("Sentence Word-Count Distribution")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Number of sentences")
axes[1].legend(fontsize=8)

# -- 3. Word count by class (box plot) ----------------------------------------
data_per_class = [df.loc[df["target"] == c, "word_count"].values for c in CLASSES]
axes[2].boxplot(data_per_class, vert=False, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[2].set_yticks(range(1, len(CLASSES) + 1))
axes[2].set_yticklabels(CLASSES)
axes[2].set_title("Sentence Length by Section Role")
axes[2].set_xlabel("Words")

plt.tight_layout()
plt.show()

# -- Example sentences per class ----------------------------------------------
print("\nExample sentences per section role:")
print("=" * 70)
for cls in CLASSES:
    sample = df.loc[df["target"] == cls, "abstract_text"].sample(2, random_state=RANDOM_STATE)
    print(f"\n{cls}:")
    for s in sample:
        print(f"  • {s[:110]}{'…' if len(s) > 110 else ''}")

## 5 · BERT Embedding Models

Six frozen BERT models span a spectrum from **general English** to **PubMed-specific**:

| Short label | HuggingFace model ID | Pretraining corpus |
|-------------|----------------------|--------------------|
| `BERT-base` | `bert-base-uncased` | General English (Wikipedia + BookCorpus) |
| `BioBERT` | `dmis-lab/biobert-base-cased-v1.1` | PubMed abstracts + PMC full-text, *on top of* BERT-base |
| `ClinicalBERT` | `medicalai/ClinicalBERT` | Clinical notes (MIMIC-III EHR) |
| `Bio_ClinicalBERT` | `emilyalsentzer/Bio_ClinicalBERT` | BioBERT initialised, then MIMIC-III |
| `PubMedBERT` | `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract` | PubMed abstracts **only**, trained from scratch |
| `SciBERT` | `allenai/scibert_scivocab_uncased` | ~3M Semantic Scholar papers (science broadly) |

**Predicted ranking on PubMed RCT sentences:**
PubMedBERT should top the leaderboard — it was trained from scratch on PubMed abstracts,
which are exactly the texts we're classifying. BioBERT (which continues BERT-base on PubMed)
should also outperform the general model. ClinicalBERT (clinical notes) and SciBERT (broad
science) sit in the middle. BERT-base is our lower-bound baseline.

Note that RCT sentence-role classification is largely a **structural** signal — METHODS sentences
have different vocabulary (verb tense, passive voice, participant counts) from RESULTS or
CONCLUSIONS regardless of the specific medical topic. Domain-specific models may still win,
but the gap could be smaller than on a topic-classification task.

**Embedding method:** mean-pool the last hidden state over all tokens (same as
`U2-1_NLP_06_OpenAlex.ipynb §6` and `U2-1_ClinicalBERT-Embeddings.ipynb`):

$$\mathbf{d} = \frac{1}{T}\sum_{t=1}^{T}\mathbf{h}_t$$

> **Slow cell note:** Embedding 6 × 300 sentences on CPU takes roughly **30–60 minutes** total.
> Set `MAX_PER_CLS = 30` (150 sentences) for a ~15-minute demo run.

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
MAX_PER_CLS = 60     # sentences per class (60 × 5 = 300 total); raise on GPU
MAX_EMB_LEN = 128    # max tokens per sentence (RCT sentences are short — rarely truncated)

BERT_MODELS = {
    "BERT-base":        "bert-base-uncased",
    "BioBERT":          "dmis-lab/biobert-base-cased-v1.1",
    "ClinicalBERT":     "medicalai/ClinicalBERT",
    "Bio_ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "PubMedBERT":       "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract",
    "SciBERT":          "allenai/scibert_scivocab_uncased",
}

# ── Balanced subset (cap per class) ──────────────────────────────────────────
np.random.seed(RANDOM_STATE)
df_sub = (
    df.groupby("target", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_PER_CLS), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)
texts  = df_sub["abstract_text"].tolist()
labels = df_sub["target"].to_numpy()

print(f"Embedding subset: {len(df_sub)} sentences  ({MAX_PER_CLS} per class)")
print(pd.Series(labels).value_counts().reindex(CLASSES).to_string())

# ── Shared train/test split on indices (same split reused for every model) ────
idx_all = np.arange(len(df_sub))
idx_tr, idx_te, y_tr, y_te = train_test_split(
    idx_all, labels, test_size=0.25, stratify=labels, random_state=RANDOM_STATE
)
print(f"\nTrain: {len(idx_tr)}   Test: {len(idx_te)}")


# ── Mean-pool embedding helper ────────────────────────────────────────────────
def embed_texts_hf(model_name, texts, max_len=MAX_EMB_LEN, log_every=25):
    """
    HuggingFace feature-extraction pipeline → mean-pool → (N, hidden_dim) array.
    Same pattern as U2-1_NLP_06_OpenAlex.ipynb §6.
    """
    pipe = hf_pipeline("feature-extraction", model=model_name)
    vecs = []
    for i, t in enumerate(texts):
        out = np.array(
            pipe(t, tokenize_kwargs={"truncation": True, "max_length": max_len})
        )[0]
        vecs.append(out.mean(axis=0))
        if (i + 1) % log_every == 0:
            print(f"    [{i+1}/{len(texts)}]")
    return np.vstack(vecs)


# ── Embed with all six models (this is the slow cell) ─────────────────────────
# Each model loads from HuggingFace Hub on first run (~400 MB, then cached).
embeddings = {}   # short_name -> np.ndarray shape (N, hidden_dim)

for short_name, model_id in BERT_MODELS.items():
    print(f"\n── {short_name}  ({model_id}) ──")
    E = embed_texts_hf(model_id, texts)
    embeddings[short_name] = E
    print(f"   Embedding shape: {E.shape}")

print("\nAll embeddings computed.")

## 6 · LR + RF on Every Embedding — Accuracy Leaderboard

We train **Logistic Regression** and **Random Forest** on each of the six embedding matrices,
using the same stratified train/test split for every combination (12 rows total).

All embeddings are `StandardScaler`-normalized before classification (LR is sensitive to
feature scale; RF is not, but normalization costs nothing and makes results more comparable).
`class_weight="balanced"` handles any minor imbalance that survives the capped sampling.

The central deliverable is a **grouped bar chart of accuracy** (and macro-F1) per model,
showing how much pretraining domain matters.

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

records      = []
predictions  = {}   # (model_label, clf_name) -> y_pred (for confusion matrix)

for model_label, E in embeddings.items():
    sc   = StandardScaler()
    E_tr = sc.fit_transform(E[idx_tr])
    E_te = sc.transform(E[idx_te])

    for clf_name, clf_proto in classifiers.items():
        clf = clone(clf_proto)
        clf.fit(E_tr, y_tr)
        y_pred = clf.predict(E_te)

        acc = accuracy_score(y_te, y_pred)
        f1  = f1_score(y_te, y_pred, average="macro", zero_division=0)

        records.append({
            "Model":       model_label,
            "Classifier":  clf_name,
            "Accuracy":    round(acc, 4),
            "Macro-F1":    round(f1,  4),
        })
        predictions[(model_label, clf_name)] = y_pred
        print(f"  {model_label:<20} | {clf_name:<22} | acc={acc:.4f}  macro-F1={f1:.4f}")

# ── Leaderboard DataFrame ─────────────────────────────────────────────────────
summary = (
    pd.DataFrame(records)
    .sort_values("Accuracy", ascending=False)
    .reset_index(drop=True)
)
print("\n── Leaderboard (sorted by Accuracy) ──")
display(summary)

# ── Grouped bar chart: accuracy + macro-F1 per (model × classifier) ──────────
model_names = list(BERT_MODELS.keys())

lr_acc = [next(r["Accuracy"] for r in records if r["Model"] == m and r["Classifier"] == "Logistic Regression") for m in model_names]
rf_acc = [next(r["Accuracy"] for r in records if r["Model"] == m and r["Classifier"] == "Random Forest")       for m in model_names]
lr_f1  = [next(r["Macro-F1"] for r in records if r["Model"] == m and r["Classifier"] == "Logistic Regression") for m in model_names]
rf_f1  = [next(r["Macro-F1"] for r in records if r["Model"] == m and r["Classifier"] == "Random Forest")       for m in model_names]

x = np.arange(len(model_names))
w = 0.20

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (lr_vals, rf_vals, metric) in zip(axes, [
        (lr_acc, rf_acc, "Accuracy"),
        (lr_f1,  rf_f1,  "Macro-F1"),
]):
    ax.bar(x - w/2, lr_vals, w, label="Logistic Regression", color="#4ec9b0")
    ax.bar(x + w/2, rf_vals, w, label="Random Forest",       color="#c586c0")
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} — LR vs RF per BERT model")
    ax.legend()
    for xi, (lv, rv) in enumerate(zip(lr_vals, rf_vals)):
        ax.text(xi - w/2, lv + 0.01, f"{lv:.2f}", ha="center", fontsize=7)
        ax.text(xi + w/2, rv + 0.01, f"{rv:.2f}", ha="center", fontsize=7)
    ax.axhline(1 / len(CLASSES), color="gray", linestyle=":", linewidth=1, label="chance")

plt.suptitle("Frozen BERT embeddings → LR / RF classifier  (6 models × 2 classifiers)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrix for the single best (model, classifier) combination ──────
best_row = summary.iloc[0]
best_key = (best_row["Model"], best_row["Classifier"])
y_pred_best = predictions[best_key]

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred_best,
    ax=ax,
    colorbar=False,
    xticks_rotation=25,
)
ax.set_title(
    f"Confusion Matrix — Best combination\n"
    f"{best_row['Model']} + {best_row['Classifier']}  "
    f"(acc={best_row['Accuracy']:.3f}, macro-F1={best_row['Macro-F1']:.3f})"
)
plt.tight_layout()
plt.show()

# ── Per-class report for best model ───────────────────────────────────────────
print(f"\nClassification report — {best_row['Model']} + {best_row['Classifier']}")
print(classification_report(y_te, y_pred_best, zero_division=0))

## 7 · t-SNE Projections — One Plot per Embedding

**t-SNE** projects high-dimensional vectors to 2-D while preserving local neighborhood
structure. If a BERT model encodes the rhetorical role of sentences, papers with the same
structural function (e.g., all METHODS sentences) should form visible clusters.

We fit a **separate t-SNE** on each full embedding matrix (all 300 sentences), color by true
section role, and arrange all six plots in a 2 × 3 grid. Each subplot's subtitle shows the
best accuracy for that model — so you can read visual separation against classification
performance side by side.

> **Teaching question:** Do the models with tighter clusters also score higher? If so,
> t-SNE cluster quality is a reliable visual proxy for embedding utility — without running
> any classifier at all.

In [ ]:
# ── Colour palette — one hex per section role ─────────────────────────────────
CLASS_PALETTE = {
    "BACKGROUND":  "#4ec9b0",   # teal
    "OBJECTIVE":   "#dcdcaa",   # gold
    "METHODS":     "#c586c0",   # purple
    "RESULTS":     "#ce9178",   # warm orange
    "CONCLUSIONS": "#5b8dd9",   # blue
}

# Pre-compute best accuracy per model (for subplot subtitles)
best_acc_per_model = {
    m: max(r["Accuracy"] for r in records if r["Model"] == m)
    for m in BERT_MODELS
}

# ── One t-SNE per embedding → 2×3 grid ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()

tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=RANDOM_STATE)

for ax, (model_label, E) in zip(axes_flat, embeddings.items()):
    coords = tsne.fit_transform(E)

    for cls in CLASSES:
        mask  = labels == cls
        color = CLASS_PALETTE[cls]
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=color, label=cls, alpha=0.72, s=20, edgecolors="none",
        )

    best_acc = best_acc_per_model[model_label]
    ax.set_title(f"{model_label}\nbest acc = {best_acc:.2f}", fontsize=10)
    ax.set_xlabel("t-SNE 1", fontsize=8)
    ax.set_ylabel("t-SNE 2", fontsize=8)
    ax.tick_params(labelsize=7)

# ── Shared legend ─────────────────────────────────────────────────────────────
handles = [mpatches.Patch(color=CLASS_PALETTE[c], label=c) for c in CLASSES]
fig.legend(handles=handles, title="Section Role", loc="lower center",
           ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.04))

plt.suptitle(
    "t-SNE projections of frozen BERT embeddings\n"
    "(colored by RCT section role — perplexity=30, init=pca)",
    fontsize=13,
)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

## 8 · Summary & Takeaways

### What the results typically show

| Model | Pretraining corpus | Expected advantage on RCT sentences? |
|---|---|---|
| `BERT-base` | General English | Baseline — no biomedical knowledge |
| `BioBERT` | PubMed + PMC (continued from BERT-base) | ✓ Direct domain match |
| `ClinicalBERT` | MIMIC-III clinical notes | ~ Partial — clinical register differs from abstracts |
| `Bio_ClinicalBERT` | BioBERT → MIMIC-III | ✓ Better than ClinicalBERT alone |
| `PubMedBERT` | PubMed abstracts only (from scratch) | ✓✓ Exact domain match, no Wikipedia noise |
| `SciBERT` | Broad science (Semantic Scholar) | ~ Marginal benefit — science vocab, not PubMed-specific |

**Key takeaways:**

1. **RCT sentence-role classification is partly structural, partly lexical.** METHODS sentences
   have distinctive passive constructions ("were randomized", "were assigned"); RESULTS sentences
   contain p-values and confidence intervals; CONCLUSIONS use hedging language. These structural
   patterns are learnable even from general BERT — so the gap between models may be smaller here
   than on a topic-classification task.

2. **PubMedBERT should still win.** Trained from scratch on PubMed abstracts, it sees exactly
   this vocabulary — abbreviations, statistical notation, clinical terminology — during pretraining.
   BioBERT (PubMed-continued) should follow closely.

3. **ClinicalBERT vs. Bio_ClinicalBERT split.** ClinicalBERT (MIMIC only) is the "mismatched
   biomedical" control — clinical notes are a different register from structured abstracts.
   Bio_ClinicalBERT starts from BioBERT and then fine-tunes on MIMIC, so it retains more
   abstract-vocabulary knowledge.

4. **The t-SNE for PubMedBERT/BioBERT should show RESULTS and METHODS pulling apart cleanly**
   — those two classes contain the most distinct vocabulary in the RCT corpus.

### Extensions & homework hooks

- **`[CLS]` pooling vs. mean pooling.** Replace `out.mean(axis=0)` with `out[0]` and re-run.
  For sentence classification the CLS token is often better — does that hold here?
- **Position feature.** Add `line_number / total_lines` as a single extra feature alongside
  the embedding. How much does knowing sentence position help (OBJECTIVE rarely appears last)?
- **Full dataset.** Set `MAX_PER_CLS = 2000` and run on GPU for a rigorous comparison.
- **Sequential context.** The current setup classifies sentences independently. Fine-tuning
  with a sequential model (LSTM or transformer over the sentence sequence) is the published
  state-of-the-art approach on this dataset.
- **Sentence-transformers baseline.** Add `all-MiniLM-L6-v2` via `SentenceTransformer` as a
  7th comparison row — it is often surprisingly competitive for short-text tasks.